# Gemma 4 31B IT: vLLM-neuron on AWS Trainium2 (SDK 2.29)

This notebook deploys **google/gemma-4-31b-it** on AWS Trainium2 (trn2.3xlarge) using **vLLM-neuron 0.5.0**
with the NxDI contrib model, then validates correctness and benchmarks throughput.

### Architecture Highlights

Gemma 4 31B is NOT natively supported in NxDI or vLLM-neuron. This notebook uses the
**contrib model** (`NeuronGemma4ForCausalLM`) and registers it into vLLM at runtime.

Key differences from Gemma 3:

| Feature | Gemma 4 31B | Gemma 3 27B |
|---------|------------|-------------|
| Global head_dim | **512** | 256 |
| SWA head_dim | 256 | 256 |
| Global KV heads | 4 | 4 |
| SWA KV heads | **16** | 1 |
| attention_k_eq_v | **Yes** (global layers) | No |
| final_logit_softcapping | **30.0** | None |
| Layer types | Explicit list (SWA/global) | Pattern-based |
| partial_rotary_factor | **0.25** (global) | N/A |

### Critical Configuration

| Setting | Value | Why |
|---------|-------|-----|
| `fused_qkv` | **False** | Heterogeneous Q/K/V shapes |
| `attn_kernel_enabled` | **False** | head_dim=512 exceeds NKI kernel limit (128) |
| `on_device_sampling_config` | **None** | Use CPU sampling for compatibility |
| `tp_degree` | **4** | 31B model on trn2.3xlarge (LNC=2, 4 cores) |

### Time to Complete

- **First run**: ~25 minutes (compilation + model load)
- **Cached run**: ~12 minutes (model load only)
- **Benchmarks**: ~5 minutes

## Prerequisites

**Instance**: trn2.3xlarge

**AMI**: `Deep Learning AMI Neuron (Ubuntu 24.04) 20260410` (SDK 2.29)

**Disk**: 300 GB EBS (gp3)

**LNC Mode**: LNC=2 (default) -- gives 4 logical cores with 24 GB HBM each

**venv**: `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/bin/activate`

**HuggingFace token**: Required for Gemma 4 model access. Set `HF_TOKEN` env var.

**Contrib model**: The `src/` directory from this contrib package must be available
on the instance (e.g., uploaded to `/home/ubuntu/gemma4-contrib/src/`).

## Step 0: Environment Detection

In [1]:
import subprocess, os, sys, json

# Check venv
VENV_PATH = '/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16'
if not os.environ.get('VIRTUAL_ENV', '').startswith(VENV_PATH):
    print(f'WARNING: Activate the vLLM venv before running this notebook:')
    print(f'  source {VENV_PATH}/bin/activate')
    print(f'  jupyter notebook')

# Detect instance type
try:
    token = subprocess.check_output(
        ['curl', '-s', '-X', 'PUT', 'http://169.254.169.254/latest/api/token',
         '-H', 'X-aws-ec2-metadata-token-ttl-seconds: 21600', '--connect-timeout', '2'],
    ).decode().strip()
    instance_type = subprocess.check_output(
        ['curl', '-s', '-H', f'X-aws-ec2-metadata-token: {token}',
         'http://169.254.169.254/latest/meta-data/instance-type', '--connect-timeout', '2'],
    ).decode().strip()
except:
    instance_type = 'unknown'
print(f'Instance: {instance_type}')

# Detect Neuron cores
neuron_ls = subprocess.check_output(['neuron-ls']).decode()
print(f'\nneuron-ls output:\n{neuron_ls}')

nc_count = neuron_ls.count('NeuronCore')
print(f'Detected {nc_count} logical NeuronCores')

if 'trn2' in instance_type:
    lnc = os.environ.get('NEURON_LOGICAL_NC_CONFIG', '2')
    print(f'LNC mode: {lnc} ({nc_count} logical cores)')

# SDK version
import torch
import torch_neuronx
print(f'\nPyTorch: {torch.__version__}')
print(f'torch-neuronx: {torch_neuronx.__version__}')

# Check HF token
hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    print(f'HF_TOKEN: set ({len(hf_token)} chars)')
else:
    print('WARNING: HF_TOKEN not set. Required for Gemma 4 model access.')
    print('Set it with: export HF_TOKEN=hf_...')

Instance: trn2.3xlarge

neuron-ls output:
instance-type: trn2.3xlarge
instance-id: i-019ecd3362dc94d7d
logical-neuroncore-config: 2
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 4      | 0-3      | 96 GB  | 0000:33:00.0 | 0-11     | 0    |
+--------+--------+----------+--------+--------------+----------+------+

Detected 0 logical NeuronCores
LNC mode: 2 (0 logical cores)

PyTorch: 2.9.1+cu128
torch-neuronx: 2.9.0.2.13.24727+8e870898
HF_TOKEN: set (37 chars)


## Step 1: Download Model and Fix Tokenizer Config

Gemma 4's `tokenizer_config.json` uses a list-format `extra_special_tokens` field
that is incompatible with transformers 4.57 (SDK 2.29). We fix it to an empty dict.

In [2]:
import os, json
from huggingface_hub import snapshot_download

MODEL_HF_ID = 'google/gemma-4-31b-it'
MODEL_LOCAL_PATH = '/home/ubuntu/models/gemma-4-31b-it'

if os.path.exists(os.path.join(MODEL_LOCAL_PATH, 'config.json')):
    print(f'Model already downloaded at {MODEL_LOCAL_PATH}')
else:
    print(f'Downloading {MODEL_HF_ID} to {MODEL_LOCAL_PATH}...')
    snapshot_download(
        MODEL_HF_ID,
        local_dir=MODEL_LOCAL_PATH,
        token=os.environ.get('HF_TOKEN'),
    )
    print('Download complete.')

# Fix tokenizer_config.json: extra_special_tokens is a list in Gemma4
# but transformers 4.57 expects a dict
tok_config_path = os.path.join(MODEL_LOCAL_PATH, 'tokenizer_config.json')
with open(tok_config_path) as f:
    tok_config = json.load(f)

est = tok_config.get('extra_special_tokens', None)
if isinstance(est, list):
    tok_config['extra_special_tokens'] = {}
    with open(tok_config_path, 'w') as f:
        json.dump(tok_config, f, indent=2, ensure_ascii=False)
    print(f'Fixed tokenizer_config.json: extra_special_tokens list -> empty dict')
else:
    print(f'tokenizer_config.json already has correct format')

# Verify model config
config_path = os.path.join(MODEL_LOCAL_PATH, 'config.json')
with open(config_path) as f:
    config = json.load(f)
text_config = config.get('text_config', config)
print(f'\nModel config:')
print(f'  model_type: {config.get("model_type", "N/A")}')
print(f'  architectures: {config.get("architectures", "N/A")}')
print(f'  hidden_size: {text_config.get("hidden_size", "N/A")}')
print(f'  num_hidden_layers: {text_config.get("num_hidden_layers", "N/A")}')
print(f'  head_dim: {text_config.get("head_dim", "N/A")}')
print(f'  global_head_dim: {text_config.get("global_head_dim", "N/A")}')
print(f'  vocab_size: {text_config.get("vocab_size", config.get("vocab_size", "N/A"))}')
print(f'  num_attention_heads: {text_config.get("num_attention_heads", "N/A")}')
print(f'  num_key_value_heads: {text_config.get("num_key_value_heads", "N/A")}')

Model already downloaded at /home/ubuntu/models/gemma-4-31b-it
tokenizer_config.json already has correct format

Model config:
  model_type: gemma4
  architectures: ['Gemma4ForConditionalGeneration']
  hidden_size: 5376
  num_hidden_layers: 60
  head_dim: 256
  global_head_dim: 512
  vocab_size: 262144
  num_attention_heads: 32
  num_key_value_heads: 16


## Step 2: Patch vLLM-neuron for Gemma4 Support

vLLM-neuron 0.5.0 does not natively support `gemma4` model_type. We apply three patches:

1. **vLLM model registry** (`registry.py`): Add `Gemma4ForConditionalGeneration` to the
   supported architectures list so `ModelConfig` validation passes.

2. **vLLM-neuron model loader** (`neuronx_distributed_model_loader.py`): Register
   `NeuronGemma4ForCausalLM` in NxDI's `MODEL_TYPES` dictionary and register a
   `_Gemma4Config` subclass of `Gemma3Config` in `AutoConfig` (handles Gemma4's nested
   `rope_parameters` format). Also add a fallback in `_get_model_configs` for configs
   with `num_key_value_heads` nested inside `text_config`.

3. **Wrapper script**: Register `_Gemma4Config` in `AutoConfig` BEFORE vLLM imports,
   so `ModelConfig.__init__` can parse the Gemma4 config.json.

In [3]:
import os, sys

# --- Configuration ---
CONTRIB_SRC = '/home/ubuntu/gemma4-contrib/src'
VENV_BASE = '/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16'
VLLM_NEURON_DIR = '/vllm/vllm_neuron'
REGISTRY_PATH = f'{VENV_BASE}/lib/python3.12/site-packages/vllm/model_executor/models/registry.py'
LOADER_PATH = f'{VLLM_NEURON_DIR}/worker/neuronx_distributed_model_loader.py'

print('=== Patch 1: Add Gemma4 to vLLM model registry ===')
with open(REGISTRY_PATH) as f:
    content = f.read()

if 'Gemma4ForConditionalGeneration' not in content:
    # Add after the Gemma3n entry
    anchor = '"Gemma3nForConditionalGeneration",\n    ),'
    if anchor in content:
        replacement = anchor + '\n    "Gemma4ForConditionalGeneration": ("gemma3", "Gemma3ForCausalLM"),  # Gemma4 contrib'
        content = content.replace(anchor, replacement)
        with open(REGISTRY_PATH, 'w') as f:
            f.write(content)
        print('  Added Gemma4ForConditionalGeneration to registry.py')
    else:
        print('  WARNING: Could not find anchor in registry.py. Manual patching needed.')
else:
    print('  Gemma4ForConditionalGeneration already in registry.py')


print('\n=== Patch 2: Register Gemma4 in NxDI model loader ===')
with open(LOADER_PATH) as f:
    loader_content = f.read()

patches_applied = 0

# Patch 2a: Register MODEL_TYPES['gemma4']
if 'MODEL_TYPES["gemma4"]' not in loader_content:
    # Insert after the MODEL_TYPES import
    anchor = 'from neuronx_distributed_inference.utils.constants import MODEL_TYPES'
    patch_block = '''from neuronx_distributed_inference.utils.constants import MODEL_TYPES
from neuronx_distributed_inference.utils.hf_adapter import load_pretrained_config
from transformers import AutoModelForCausalLM, PretrainedConfig

# Register Gemma4 contrib model if available on sys.path
if "gemma4" not in MODEL_TYPES:
    try:
        from modeling_gemma4 import NeuronGemma4ForCausalLM

        MODEL_TYPES["gemma4"] = {"causal-lm": NeuronGemma4ForCausalLM}
        logging.getLogger(__name__).info(
            "Registered Gemma4 contrib model in MODEL_TYPES"
        )
    except ImportError:
        logging.getLogger(__name__).debug(
            "Gemma4 contrib model not found on sys.path, skipping registration"
        )

# Register Gemma4Config in transformers AutoConfig for config.json parsing
try:
    from transformers import AutoConfig
    from transformers.models.gemma3.configuration_gemma3 import Gemma3Config

    class _Gemma4Config(Gemma3Config):
        model_type = "gemma4"

        def __init__(self, **kwargs):
            rp = kwargs.get("rope_parameters", None)
            if rp and isinstance(rp, dict) and "rope_type" not in rp:
                self._gemma4_rope_parameters = rp
                if "sliding_attention" in rp:
                    kwargs["rope_parameters"] = rp["sliding_attention"]
                elif "full_attention" in rp:
                    kwargs["rope_parameters"] = rp["full_attention"]
            tc = kwargs.get("text_config", None)
            if tc and isinstance(tc, dict):
                tc_rp = tc.get("rope_parameters", None)
                if tc_rp and isinstance(tc_rp, dict) and "rope_type" not in tc_rp:
                    if "sliding_attention" in tc_rp:
                        tc["rope_parameters"] = tc_rp["sliding_attention"]
                    elif "full_attention" in tc_rp:
                        tc["rope_parameters"] = tc_rp["full_attention"]
            super().__init__(**kwargs)
            if hasattr(self, "_gemma4_rope_parameters"):
                self.rope_parameters = self._gemma4_rope_parameters

    try:
        AutoConfig.register("gemma4", _Gemma4Config)
    except ValueError:
        pass  # Already registered
except Exception:
    pass'''
    # Replace the anchor + next two lines
    old_section = '''from neuronx_distributed_inference.utils.constants import MODEL_TYPES
from neuronx_distributed_inference.utils.hf_adapter import load_pretrained_config
from transformers import AutoModelForCausalLM, PretrainedConfig'''
    if old_section in loader_content:
        loader_content = loader_content.replace(old_section, patch_block)
        patches_applied += 1
        print('  Added MODEL_TYPES["gemma4"] and _Gemma4Config registration')
    else:
        print('  WARNING: Could not find anchor for MODEL_TYPES patch')
else:
    print('  MODEL_TYPES["gemma4"] already registered')

# Patch 2b: Add text_config fallback in _get_model_configs
if 'Fallback: if top-level' not in loader_content:
    old_configs = '''    if architecture in NEURON_MULTI_MODAL_MODELS:
        config = getattr(config, "text_config", None)
    num_key_value_heads = getattr(config, "num_key_value_heads", None)'''
    new_configs = '''    if architecture in NEURON_MULTI_MODAL_MODELS:
        config = getattr(config, "text_config", None)
    num_key_value_heads = getattr(config, "num_key_value_heads", None)
    # Fallback: if top-level config doesn't have num_key_value_heads, check text_config
    # (handles models like Gemma4 with nested text_config but not in NEURON_MULTI_MODAL_MODELS)
    if num_key_value_heads is None and hasattr(config, "text_config"):
        config = config.text_config
        num_key_value_heads = getattr(config, "num_key_value_heads", None)'''
    if old_configs in loader_content:
        loader_content = loader_content.replace(old_configs, new_configs)
        patches_applied += 1
        print('  Added text_config fallback in _get_model_configs')
    else:
        print('  WARNING: Could not find anchor for _get_model_configs patch')
else:
    print('  text_config fallback already present')

if patches_applied > 0:
    with open(LOADER_PATH, 'w') as f:
        f.write(loader_content)
    print(f'  Wrote {patches_applied} patches to model loader')

# Clear __pycache__ to ensure patches take effect
import shutil
for cache_dir in [
    f'{VENV_BASE}/lib/python3.12/site-packages/vllm/model_executor/models/__pycache__',
    f'{VLLM_NEURON_DIR}/__pycache__',
    f'{VLLM_NEURON_DIR}/worker/__pycache__',
]:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
print('\nCleared __pycache__ directories')

print('\n=== Patch 3: Create wrapper script ===')
# The wrapper registers _Gemma4Config in AutoConfig BEFORE vLLM imports.
# This is needed because ModelConfig.__init__ calls AutoConfig.from_pretrained()
# before the NxDI model loader (which also registers it) is imported.
wrapper_path = '/home/ubuntu/vllm_gemma4_wrapper.py'
wrapper_code = '''"""Wrapper to register Gemma4Config in transformers before vLLM imports."""\nimport sys\n\nfrom transformers import AutoConfig\nfrom transformers.models.gemma3.configuration_gemma3 import Gemma3Config\n\nclass _Gemma4Config(Gemma3Config):\n    model_type = "gemma4"\n\n    def __init__(self, **kwargs):\n        rp = kwargs.get("rope_parameters", None)\n        if rp and isinstance(rp, dict) and "rope_type" not in rp:\n            self._gemma4_rope_parameters = rp\n            if "sliding_attention" in rp:\n                kwargs["rope_parameters"] = rp["sliding_attention"]\n            elif "full_attention" in rp:\n                kwargs["rope_parameters"] = rp["full_attention"]\n        tc = kwargs.get("text_config", None)\n        if tc and isinstance(tc, dict):\n            tc_rp = tc.get("rope_parameters", None)\n            if tc_rp and isinstance(tc_rp, dict) and "rope_type" not in tc_rp:\n                if "sliding_attention" in tc_rp:\n                    tc["rope_parameters"] = tc_rp["sliding_attention"]\n                elif "full_attention" in tc_rp:\n                    tc["rope_parameters"] = tc_rp["full_attention"]\n        super().__init__(**kwargs)\n        if hasattr(self, "_gemma4_rope_parameters"):\n            self.rope_parameters = self._gemma4_rope_parameters\n\ntry:\n    AutoConfig.register("gemma4", _Gemma4Config)\n    print("[Gemma4] Registered _Gemma4Config in AutoConfig", flush=True)\nexcept ValueError:\n    print("[Gemma4] _Gemma4Config already registered", flush=True)\n\nimport runpy\nsys.argv[0] = "vllm.entrypoints.openai.api_server"\nrunpy.run_module("vllm.entrypoints.openai.api_server", run_name="__main__", alter_sys=True)\n'''
with open(wrapper_path, 'w') as f:
    f.write(wrapper_code)
print(f'  Wrapper written to {wrapper_path}')

print('\nAll patches applied successfully.')

=== Patch 1: Add Gemma4 to vLLM model registry ===
  Gemma4ForConditionalGeneration already in registry.py

=== Patch 2: Register Gemma4 in NxDI model loader ===
  MODEL_TYPES["gemma4"] already registered
  text_config fallback already present

Cleared __pycache__ directories

=== Patch 3: Create wrapper script ===
  Wrapper written to /home/ubuntu/vllm_gemma4_wrapper.py

All patches applied successfully.


## Step 3: Configure and Launch vLLM-neuron

### Configuration Notes

| Setting | Value | Why |
|---------|-------|-----|
| `fused_qkv` | **false** | Q has 32 heads, K has 4/16, V has 4/16 -- cannot fuse |
| `attn_kernel_enabled` | **false** | NKI kernel limit is head_dim=128, we have 512 |
| `on_device_sampling_config` | **null** | CPU sampling for compatibility |
| `batch_size` | **1** | 31B model -- memory constrained |
| `n_active_tokens` | **1** | Must match batch_size |
| `seq_len` | **4096** | Reasonable context window |
| `tp_degree` | **4** | Full trn2.3xlarge (LNC=2) |
| `context_encoding_buckets` | **[512, 1024, 2048, 4096]** | Multiple CTE buckets |
| `token_generation_buckets` | **[4096]** | TKG bucket matches seq_len |

In [4]:
import os, json

# --- Model and serving configuration ---
MODEL_PATH = '/home/ubuntu/models/gemma-4-31b-it'
MAX_MODEL_LEN = 4096
MAX_NUM_SEQS = 1
DTYPE = 'bfloat16'
TP_DEGREE = 4
BATCH_SIZE = 1
PORT = 8000

# --- Neuron-specific override config ---
NEURON_OVERRIDE_CONFIG = {
    'override_neuron_config': {
        'tp_degree': TP_DEGREE,
        'batch_size': BATCH_SIZE,
        'seq_len': MAX_MODEL_LEN,
        'n_active_tokens': BATCH_SIZE,
        'context_encoding_buckets': [512, 1024, 2048, 4096],
        'token_generation_buckets': [4096],
        'on_device_sampling_config': None,
        'attn_kernel_enabled': False,
        'fused_qkv': False,
    }
}

print('Neuron override config:')
print(json.dumps(NEURON_OVERRIDE_CONFIG, indent=2, default=str))

print(f'\nServer will listen on port {PORT}')
print(f'Model: {MODEL_PATH}')
print(f'Batch size: {BATCH_SIZE} | max_model_len: {MAX_MODEL_LEN} | TP: {TP_DEGREE}')

Neuron override config:
{
  "override_neuron_config": {
    "tp_degree": 4,
    "batch_size": 1,
    "seq_len": 4096,
    "n_active_tokens": 1,
    "context_encoding_buckets": [
      512,
      1024,
      2048,
      4096
    ],
    "token_generation_buckets": [
      4096
    ],
    "on_device_sampling_config": null,
    "attn_kernel_enabled": false,
    "fused_qkv": false
  }
}

Server will listen on port 8000
Model: /home/ubuntu/models/gemma-4-31b-it
Batch size: 1 | max_model_len: 4096 | TP: 4


In [5]:
import subprocess, os, json

# Build the --additional-config JSON string
additional_config_str = json.dumps(NEURON_OVERRIDE_CONFIG)

# Write launch script that uses the wrapper for early config registration
launch_script = f"""#!/bin/bash
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/bin/activate
export VLLM_NEURON_FRAMEWORK=neuronx-distributed-inference
export VLLM_RPC_TIMEOUT=300000
export HF_TOKEN={os.environ.get('HF_TOKEN', '')}
export NEURON_CC_FLAGS='--retry_failed_compilation'
export PYTHONPATH={CONTRIB_SRC}:${{PYTHONPATH:-}}

python3 /home/ubuntu/vllm_gemma4_wrapper.py \\
  --model {MODEL_PATH} \\
  --tensor-parallel-size {TP_DEGREE} \\
  --max-model-len {MAX_MODEL_LEN} \\
  --max-num-seqs {MAX_NUM_SEQS} \\
  --dtype {DTYPE} \\
  --port {PORT} \\
  --disable-log-requests \\
  --no-enable-prefix-caching \\
  --additional-config '{additional_config_str}'
"""

script_path = '/home/ubuntu/launch_vllm_gemma4.sh'
with open(script_path, 'w') as f:
    f.write(launch_script)
os.chmod(script_path, 0o755)
print(f'Launch script written to {script_path}')

# Kill any existing vLLM server
subprocess.run(['pkill', '-9', '-f', 'vllm'], capture_output=True)
import time; time.sleep(3)

# Launch in background
log_path = '/home/ubuntu/vllm_gemma4.log'
log_file = open(log_path, 'w')
server_proc = subprocess.Popen(
    ['/bin/bash', script_path],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print(f'Server PID: {server_proc.pid}')
print(f'Log: {log_path}')
print(f'\nCompilation will take ~15-25 min on first run, ~2-5 min from cache.')

Launch script written to /home/ubuntu/launch_vllm_gemma4.sh
Server PID: 5389
Log: /home/ubuntu/vllm_gemma4.log

Compilation will take ~15-25 min on first run, ~2-5 min from cache.


In [6]:
import time, requests

print('Waiting for vLLM-neuron to compile and start...')
start = time.time()
timeout = 3600  # 60 minutes max (31B model is large)

while time.time() - start < timeout:
    # Check if process died
    if server_proc.poll() is not None:
        print(f'\nERROR: Server process died with code {server_proc.returncode}')
        print('Last 50 lines of log:')
        with open('/home/ubuntu/vllm_gemma4.log') as f:
            lines = f.readlines()
            print(''.join(lines[-50:]))
        raise RuntimeError('vLLM server failed to start')
    
    try:
        resp = requests.get(f'http://localhost:{PORT}/health', timeout=2)
        if resp.status_code == 200:
            elapsed = time.time() - start
            print(f'\nvLLM-neuron is READY! ({elapsed:.0f}s = {elapsed/60:.1f} min)')
            break
    except:
        pass
    
    time.sleep(15)
    elapsed = time.time() - start
    if int(elapsed) % 60 < 15:
        with open('/home/ubuntu/vllm_gemma4.log') as f:
            lines = f.readlines()
            # Count compiler passes
            passes = sum(1 for l in lines if 'Compiler status PASS' in l)
            last_line = lines[-1].strip() if lines else '(no output)'
        print(f'  [{elapsed/60:.0f} min] Compiler passes: {passes}/6 | {last_line[:100]}')
else:
    print(f'TIMEOUT after {timeout/60:.0f} min')
    print('Last 30 lines of log:')
    with open('/home/ubuntu/vllm_gemma4.log') as f:
        lines = f.readlines()
        print(''.join(lines[-30:]))
    raise TimeoutError('vLLM-neuron failed to start')

# Verify model listing
resp = requests.get(f'http://localhost:{PORT}/v1/models')
models = resp.json()
MODEL_ID = models['data'][0]['id']
print(f'Serving model: {MODEL_ID}')

Waiting for vLLM-neuron to compile and start...
  [1 min] Compiler passes: 0/6 | (APIServer pid=5390) INFO 04-24 03:44:18 [utils.py:223] non-default args: {'model': '/home/ubuntu/mo
  [2 min] Compiler passes: 0/6 | (EngineCore_DP0 pid=5528)   with torch.cuda.amp.autocast(enabled=False):
  [3 min] Compiler passes: 0/6 | ..
  [4 min] Compiler passes: 0/6 | .....
  [5 min] Compiler passes: 0/6 | ........
  [6 min] Compiler passes: 0/6 | ...........
  [7 min] Compiler passes: 0/6 | ..............
  [8 min] Compiler passes: 1/6 | ....
  [9 min] Compiler passes: 1/6 | ................
  [10 min] Compiler passes: 4/6 | .
  [11 min] Compiler passes: 4/6 | ....
  [12 min] Compiler passes: 5/6 | ^
  [13 min] Compiler passes: 5/6 | ..
  [14 min] Compiler passes: 5/6 | .....
  [15 min] Compiler passes: 5/6 | ........
  [16 min] Compiler passes: 6/6 | (EngineCore_DP0 pid=5528) [2026-04-24 03:59:15.719: I neuronx_distributed/parallel_layers/parallel_s

vLLM-neuron is READY! (1005s = 16.8 min)
Servin

## Step 4: Validate Model Output

Test with known prompts that were validated in the integration tests.

In [7]:
import requests, json, time

url = f'http://localhost:{PORT}/v1/chat/completions'

# Test 1: Simple math
payload = {
    'model': MODEL_ID,
    'messages': [{'role': 'user', 'content': 'What is 2+2? Answer with just the number.'}],
    'max_tokens': 10,
    'temperature': 0.0,
}

start = time.perf_counter()
resp = requests.post(url, json=payload)
elapsed = (time.perf_counter() - start) * 1000
result = resp.json()

if 'choices' in result:
    text = result['choices'][0]['message']['content']
    usage = result.get('usage', {})
    print(f'Test 1 (Math): "{text}"')
    print(f'  Prompt tokens: {usage.get("prompt_tokens", "N/A")}')
    print(f'  Completion tokens: {usage.get("completion_tokens", "N/A")}')
    print(f'  Latency: {elapsed:.1f}ms')
    assert '4' in text, f'Expected 4 in response, got: {text}'
else:
    print(f'ERROR: {json.dumps(result, indent=2)}')

# Test 2: Knowledge
payload['messages'] = [{'role': 'user', 'content': 'What is the capital of France? One word answer.'}]

start = time.perf_counter()
resp = requests.post(url, json=payload)
elapsed = (time.perf_counter() - start) * 1000
result = resp.json()

if 'choices' in result:
    text = result['choices'][0]['message']['content']
    print(f'\nTest 2 (Knowledge): "{text}"')
    print(f'  Latency: {elapsed:.1f}ms')
    assert 'paris' in text.lower(), f'Expected Paris in response, got: {text}'
else:
    print(f'ERROR: {json.dumps(result, indent=2)}')

# Test 3: Longer generation
payload['messages'] = [{'role': 'user', 'content': 'Write a haiku about the ocean.'}]
payload['max_tokens'] = 64

start = time.perf_counter()
resp = requests.post(url, json=payload)
elapsed = (time.perf_counter() - start) * 1000
result = resp.json()

if 'choices' in result:
    text = result['choices'][0]['message']['content']
    usage = result.get('usage', {})
    print(f'\nTest 3 (Haiku): "{text}"')
    print(f'  Completion tokens: {usage.get("completion_tokens", "N/A")}')
    print(f'  Latency: {elapsed:.1f}ms')
    assert len(text.split()) > 3, f'Expected multi-word response, got: {text}'
else:
    print(f'ERROR: {json.dumps(result, indent=2)}')

print('\nAll validation tests passed!')

Test 1 (Math): "4"
  Prompt tokens: 26
  Completion tokens: 2
  Latency: 702.2ms

Test 2 (Knowledge): "Paris"
  Latency: 198.9ms

Test 3 (Haiku): "Blue waves kiss the shore,
Endless depths of salt and mist,
Tides drift in and out."
  Completion tokens: 24
  Latency: 962.5ms

All validation tests passed!


## Step 5: Benchmark

Measure end-to-end latency and throughput at concurrency=1 (BS=1 model).

3 warmup + 5 measured runs per workload.

In [8]:
import time, json, requests, numpy as np

VLLM_URL = f'http://localhost:{PORT}/v1/chat/completions'
WARMUP_RUNS = 3
MEASURED_RUNS = 5

SHORT_PROMPT = 'What is the capital of France? Please explain in detail.'
LONG_PROMPT = ('Write a comprehensive essay about the history of artificial intelligence, '
    'covering its origins in the 1950s, the AI winters, the resurgence of neural networks, '
    'deep learning breakthroughs, and the current state of large language models. '
    'Include discussion of key researchers, landmark papers, and pivotal moments. ') * 20

workloads = [
    ('short-short', SHORT_PROMPT, 128),
    ('short-long',  SHORT_PROMPT, 512),
    ('long-short',  LONG_PROMPT,  128),
    ('long-long',   LONG_PROMPT,  512),
]

print('=' * 80)
print(f'Gemma 4 31B Benchmark on {instance_type}')
print(f'TP: {TP_DEGREE} | BS: {BATCH_SIZE} | max_model_len: {MAX_MODEL_LEN}')
print(f'Warmup: {WARMUP_RUNS} | Measured: {MEASURED_RUNS}')
print('=' * 80)

all_results = []

for name, prompt, max_tokens in workloads:
    print(f'\n--- {name} (max_tokens={max_tokens}) ---')
    
    payload = {
        'model': MODEL_ID,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
        'temperature': 0.0,
    }
    
    # Warmup
    for i in range(WARMUP_RUNS):
        resp = requests.post(VLLM_URL, json=payload, timeout=300)
        if resp.status_code != 200:
            print(f'  Warmup {i} failed: {resp.status_code}')
    
    # Measured runs
    latencies = []
    prompt_tokens_list = []
    gen_tokens_list = []
    
    for i in range(MEASURED_RUNS):
        t0 = time.perf_counter()
        resp = requests.post(VLLM_URL, json=payload, timeout=300)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        
        if resp.status_code == 200:
            data = resp.json()
            usage = data.get('usage', {})
            latencies.append(elapsed_ms)
            prompt_tokens_list.append(usage.get('prompt_tokens', 0))
            gen_tokens_list.append(usage.get('completion_tokens', 0))
        else:
            print(f'  Run {i} failed: {resp.status_code}')
    
    if not latencies:
        print('  No successful runs!')
        continue
    
    latencies = np.array(latencies)
    avg_prompt = np.mean(prompt_tokens_list)
    avg_gen = np.mean(gen_tokens_list)
    avg_e2e = np.mean(latencies)
    estimated_tok_s = avg_gen / (avg_e2e / 1000) if avg_e2e > 0 else 0
    
    stats = {
        'workload': name,
        'avg_prompt_tokens': round(float(avg_prompt)),
        'avg_gen_tokens': round(float(avg_gen)),
        'e2e_avg_ms': round(float(np.mean(latencies)), 1),
        'e2e_p50_ms': round(float(np.percentile(latencies, 50)), 1),
        'e2e_p95_ms': round(float(np.percentile(latencies, 95)), 1),
        'est_tok_s': round(float(estimated_tok_s), 1),
    }
    all_results.append(stats)
    
    print(f'  Prompt: ~{stats["avg_prompt_tokens"]} tok | Gen: ~{stats["avg_gen_tokens"]} tok')
    print(f'  E2E: avg={stats["e2e_avg_ms"]:.1f}ms  p50={stats["e2e_p50_ms"]:.1f}ms  p95={stats["e2e_p95_ms"]:.1f}ms')
    print(f'  Est throughput: {stats["est_tok_s"]:.1f} tok/s')

print('\nBenchmark complete!')

Gemma 4 31B Benchmark on trn2.3xlarge
TP: 4 | BS: 1 | max_model_len: 4096
Warmup: 3 | Measured: 5

--- short-short (max_tokens=128) ---
  Prompt: ~25 tok | Gen: ~128 tok
  E2E: avg=4593.2ms  p50=4592.7ms  p95=4598.8ms
  Est throughput: 27.9 tok/s

--- short-long (max_tokens=512) ---
  Prompt: ~25 tok | Gen: ~468 tok
  E2E: avg=16401.7ms  p50=16405.0ms  p95=16440.1ms
  Est throughput: 28.5 tok/s

--- long-short (max_tokens=128) ---
  Prompt: ~1193 tok | Gen: ~128 tok
  E2E: avg=4833.3ms  p50=4832.3ms  p95=4845.0ms
  Est throughput: 26.5 tok/s

--- long-long (max_tokens=512) ---
  Prompt: ~1193 tok | Gen: ~512 tok
  E2E: avg=18195.2ms  p50=18186.5ms  p95=18254.4ms
  Est throughput: 28.1 tok/s

Benchmark complete!


## Step 6: Results Summary

In [9]:
import json
from datetime import datetime

# Save results
output = {
    'benchmark': 'gemma4-31b-vllm',
    'sdk_version': '2.29',
    'instance_type': instance_type,
    'accelerator': 'AWS Trainium2',
    'serving': 'vLLM-neuron 0.5.0 + Gemma4 contrib model',
    'model': 'google/gemma-4-31b-it',
    'dtype': DTYPE,
    'tp_degree': TP_DEGREE,
    'batch_size': BATCH_SIZE,
    'max_model_len': MAX_MODEL_LEN,
    'max_num_seqs': MAX_NUM_SEQS,
    'timestamp': datetime.now().isoformat(),
    'results': all_results,
}

outfile = f'/home/ubuntu/gemma4_vllm_benchmark_{instance_type}.json'
with open(outfile, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results saved to {outfile}')

# Summary table
print('\n' + '=' * 90)
print(f'SUMMARY: Gemma 4 31B on {instance_type} (SDK 2.29, vLLM-neuron, BF16, TP={TP_DEGREE})')
print('=' * 90)
header = f'{"Workload":>12} | {"In tok":>7} | {"Out tok":>8} | {"E2E avg":>10} | {"E2E p50":>10} | {"E2E p95":>10} | {"Est tok/s":>10}'
print(header)
print('-' * len(header))
for r in all_results:
    print(f'{r["workload"]:>12} | {r["avg_prompt_tokens"]:>5}   | {r["avg_gen_tokens"]:>6}   | '
          f'{r["e2e_avg_ms"]:>8.1f}ms | {r["e2e_p50_ms"]:>8.1f}ms | '
          f'{r["e2e_p95_ms"]:>8.1f}ms | {r["est_tok_s"]:>8.1f}')

print('\n--- Comparison with NxDI Direct (from integration tests) ---')
print(f'NxDI direct (seq_len=512):  TTFT=65ms, TPOT=31ms, 32.4 tok/s')
print(f'NxDI direct (seq_len=4096): TTFT=999ms, TPOT=31ms, 32.0 tok/s')
print(f'vLLM adds serving overhead but enables continuous batching and OpenAI API.')

Results saved to /home/ubuntu/gemma4_vllm_benchmark_trn2.3xlarge.json

SUMMARY: Gemma 4 31B on trn2.3xlarge (SDK 2.29, vLLM-neuron, BF16, TP=4)
    Workload |  In tok |  Out tok |    E2E avg |    E2E p50 |    E2E p95 |  Est tok/s
-------------------------------------------------------------------------------------
 short-short |    25   |    128   |   4593.2ms |   4592.7ms |   4598.8ms |     27.9
  short-long |    25   |    468   |  16401.7ms |  16405.0ms |  16440.1ms |     28.5
  long-short |  1193   |    128   |   4833.3ms |   4832.3ms |   4845.0ms |     26.5
   long-long |  1193   |    512   |  18195.2ms |  18186.5ms |  18254.4ms |     28.1

--- Comparison with NxDI Direct (from integration tests) ---
NxDI direct (seq_len=512):  TTFT=65ms, TPOT=31ms, 32.4 tok/s
NxDI direct (seq_len=4096): TTFT=999ms, TPOT=31ms, 32.0 tok/s
vLLM adds serving overhead but enables continuous batching and OpenAI API.


## Cleanup

In [10]:
import signal

try:
    server_proc.send_signal(signal.SIGTERM)
    server_proc.wait(timeout=10)
    print('vLLM server stopped.')
except:
    server_proc.kill()
    print('vLLM server killed.')

print('\nDon\'t forget to terminate the instance when done!')
print(f'Instance type: {instance_type}')

vLLM server stopped.

Don't forget to terminate the instance when done!
Instance type: trn2.3xlarge


## Appendix: Compatibility Patches Explained

Gemma 4 requires several patches to work with vLLM-neuron 0.5.0 (SDK 2.29) because:

### 1. `model_type: gemma4` not in transformers 4.57

Gemma 4 needs transformers >= 5.5.0, but SDK 2.29 ships 4.57. We create a
`_Gemma4Config` subclass of `Gemma3Config` with `model_type = "gemma4"` and
register it via `AutoConfig.register()`. The `__init__` override handles Gemma4's
nested `rope_parameters` format (per-attention-type dicts) that Gemma3Config
doesn't expect.

### 2. `Gemma4ForConditionalGeneration` not in vLLM model registry

vLLM validates architectures against a hardcoded dict in `registry.py` during
`ModelConfig.__init__`. We add `Gemma4ForConditionalGeneration` mapped to the
text-only `gemma3` module (the NxDI model loader intercepts before this mapping
is used).

### 3. `num_key_value_heads` nested in `text_config`

Gemma4's HF config has `num_key_value_heads` inside `text_config`, not at the
top level. The NxDI model loader's `_get_model_configs` only descends into
`text_config` for models in `NEURON_MULTI_MODAL_MODELS`. We add a fallback
that checks `text_config` when the top-level value is missing.

### 4. `tokenizer_config.json` format incompatibility

Gemma4's tokenizer has `extra_special_tokens` as a list, but transformers 4.57
expects a dict. We convert it to an empty dict.

### Config Flow

1. Wrapper registers `_Gemma4Config` in `AutoConfig` before vLLM imports
2. vLLM reads `config.json` -> `model_type: gemma4` -> uses `_Gemma4Config`
3. `ModelConfig` validates `architectures: ["Gemma4ForConditionalGeneration"]` against registry
4. NxDI model loader: `_get_neuron_model_cls("Gemma4ForConditionalGeneration")`
   splits at "For" -> `model="gemma4"`, `task="ConditionalGeneration"` -> `"causal-lm"`
5. Looks up `MODEL_TYPES["gemma4"]["causal-lm"]` -> `NeuronGemma4ForCausalLM`
6. Model compiles and loads via NxDI